# OCR a ticker's filings

## 1 · Parameters — the only cell you edit

In [ ]:
# ── PARAMETERS — the only cell you edit ───────────────────────────────
ENVIRONMENT = "KAGGLE"        # "LOCAL" = parse here | "KAGGLE" = ship it to a T4
EXCHANGE    = "HOSE"         # HOSE | HNX | UPCOM
SYMBOL      = "FPT"          # ticker, as CafeF files it

# WHICH QUARTERS — A LIST, AND NOTHING ELSE. Each entry is YYYY-QQ; "2026-Q4" and the
# zero-padded "2026-04" are the same quarter, folded once at the edge.
#   []                     ->  EVERY quarter this ticker files  (⚠️ ~70 documents, hours).
#                              ⚠️ Safe on this 4 GiB card ONLY because `ISOLATE_DOCUMENTS`
#                              is on; the same list in one process died at document 4
#                              (`GPU-1`). `ONLY_MISSING` below narrows it to the gap.
#   ["2014-Q4", "2015-03"] ->  exactly these quarters, and nothing else.
# ⚠️ The repo-native "Q3-2014" is REFUSED rather than quietly accepted: a typo has to report
#    itself as a typo, not as a quarter CafeF does not file. A STRING is refused for the same
#    reason — a bare "2014-Q4" with the brackets forgotten included; §2 has the measurement.
QUARTERS = []   # ⚠️ THE DEFAULT CHANGED 2026-09-03 AND IT IS THE EXPENSIVE DIRECTION. This
                # read "OUTSTANDING", which resolved to the GAP and raised when there was none;
                # `[]` is every quarter the ticker files — ~70 documents and hours — on a
                # notebook somebody just pressed run on. §2 and §3 both print which it is and
                # how many documents, before anything is spent.
                #   the old default back:  ONLY_MISSING = True
                #   one quarter:           QUARTERS = ["2014-Q4"]

# ⚠️ NARROW AN EMPTY `QUARTERS` TO WHAT IS STILL MISSING — read ONLY when the list is empty,
#    and it is what the retired "OUTSTANDING" sentinel became.
#   False -> every quarter the ticker files, which is what `[]` says above.
#   True  -> exactly the quarters §3 finds still `missing` AND still winnable, plus the span
#            operands they need. Resolved from the three statement CSVs and the PDF index,
#            printed before anything is spent, and it RAISES rather than falling through to
#            "every quarter" when there is nothing left to do.
ONLY_MISSING = False

# ⚠️ What to do about a quarter ALREADY on disk:
#   False -> FILL THE GAPS. One reading `pdf` in all three statements is dropped before any OCR
#            (and before it is uploaded); a figure that DIFFERS is never written over it.
#   True  -> re-parse every selected quarter and let the result replace what disk holds.
#   ⚠️ To replace ONE wrong row use REPAIR below, never this — see the note there.
# ⚠️ TRUE IS REQUIRED BY `SPAN_OPERANDS`, and that is the only reason it is the default here:
#    a span operand is BY DEFINITION a quarter already reading `pdf`, so with False it is
#    dropped before any OCR and the Q4 it unblocks stays unwritable. It is safe in this
#    combination ONLY because MERGE_INTO_CSV is off — `force_differs` follows OVERWRITE into
#    the automatic per-quarter merge, and never into §9's.
OVERWRITE = True

# UPSERT the accepted statements into raw_data/.../statements/*.csv, through `pdf_ocr_merge`:
# it BACKS THE THREE CSVs UP FIRST, prints every changed cell, and refuses four things it
# cannot judge — a statement whose `sane` band was empty, a figure that DIFFERS from a good
# `pdf` row, a cumulative income statement whose priors it cannot subtract, and ⚠️ a document
# any of whose layers RAISED (`VCR-1`: an exception measures the MACHINE, not the filing, so
# whatever won the cascade won by default).
# ⚠️ OFF, AND §9 DOES THE UPSERT — for two independent reasons:
#    (1) the automatic path passes `force_differs = OVERWRITE`, which is True above;
#    (2) `merge_run` PLANS THE WHOLE FOLDER AGAINST DISK AND WRITES AFTERWARDS, so one call
#        would decide a Q4 while the Q3 span it depends on is still whatever disk held when
#        the call started. §9 merges one period at a time, oldest first, which is the only
#        shape in which a span operand reaches the quarter it exists to unblock.
# ⚠️ OFF NO LONGER MEANS "THE CSVs ARE LEFT ALONE" — §9 WRITES BY DEFAULT since
#    2026-09-04 (`MERGE_APPLY = True` below). What this flag decides now is only WHICH
#    path does the upsert: the pull's one blanket call with `force_differs = OVERWRITE`,
#    or §9's ordered unforced one. Leave it off — §9 is the safer of the two, not the
#    slower one.
MERGE_INTO_CSV = False

# ⚠️ BOOTSTRAP A TICKER THAT HAS NO STATEMENT CSV YET — and it lifts a real guard (`BND-1`).
#   True  -> write a statement whose `sane` band was EMPTY. The ONLY way a new ticker starts.
#   False -> keep the guard. Correct for a ticker that already has history on disk.
# ⚠️ TRUE, AND IT IS NOT OPTIONAL FOR THIS TICKER — FPT HAD NO STATEMENT CSV AT ALL
#    (`BND-1`). `seed_history` rebuilds `sane`'s band from the `pdf` rows ON DISK, so with
#    no CSV the band is EMPTY for every document — the artefact of 2026-09-04 records
#    `history_sizes` of 0/0/0 on all three statements — and `pdf_ocr_merge`'s refusal 2
#    then skips EVERY statement the run accepted. The loop closes on itself: no CSV -> an
#    empty band -> nothing written -> still no CSV. Measured here: the 2026-09-04 Q1-2026
#    run accepted 2 of 3 statements on a T4 and wrote 0 of them, and this single flag is
#    the whole difference between `0 statement(s) would be written` and `2`.
# ⚠️ IT LIFTS A REAL GUARD, so what replaces it is ARITHMETIC, run over the artefact before
#    anything was written (no PDF, no OCR, no network) — and Q1-2026 passed all four:
#      A 41,527,873,060,120 + B 27,058,221,725,097 = TỔNG CỘNG TÀI SẢN = TỔNG CỘNG NGUỒN
#        VỐN = 68,586,094,785,217                                          residual 0
#      opening 10,522,105,729,992 + net -2,516,517,314,174 + fx -12,010,804,176
#        = closing 7,993,577,611,642                                       residual 0
#      I + II + III = the net movement                                     residual 0
#      ⚠️ and the INDEPENDENT one: the balance sheet's own cash line (5,022,685,693,032 +
#        2,970,891,918,610) = 7,993,577,611,642 = the cash flow's closing balance. TWO
#        STATEMENTS AGREEING ON ONE FIGURE TO THE ĐỒNG is the strongest check available
#        without `sane`, and it is what `BND-1`'s procedure asks for (CLAUDE.md
#        §6-2-duodequinquagies, §6-2-quattuortricies).
# ⚠️ SET IT BACK TO False once FPT has history on disk — from then on the guard can run,
#    and leaving this on would keep it switched off for quarters that no longer need it.
FORCE_EMPTY_BAND = True

# ⚠️ THE ONNX-ONLY CASCADE — 53 layers of 55, and it is about REPRODUCING, not about speed.
#   True  -> drop `tesseract@200` and `tesseract@400+relax`.
#   False -> the full cascade as shipped.
# ⚠️ `tesseract@200` IS LAYER 4 OF 55 HERE AND DOES NOT EXIST ON A KAGGLE WORKER (`TSS-1`,
#    CLAUDE.md §6-2-quinquagies). So it can win a statement twenty onnx layers would have read
#    better, and every ticker bootstrapped on a T4 carries rows produced by the 53-layer
#    cascade — a local re-parse under the full 55 is a DIFFERENT PROCEDURE and reports the
#    difference as DIFFERS. Measured on BSR Q3-2019: `tesseract@200` read
#    361,884,738 where the Kaggle `onnx@300+tail` row reads 361,884,738,267.
ONNX_ONLY = True

# ⚠️ PULL IN THE QUARTERS A CUMULATIVE Q4 NEEDS AS OPERANDS (`QUARTERS = []` with
#    ONLY_MISSING = True only — the other two modes already name every quarter they are going
#    to open, so there is nothing left for this to add).
#   A Q4 income statement is the YEAR, and the standalone quarter is FY − (Q1+Q2+Q3). The
#   merge will only subtract a prior whose span is a KNOWN three months, and most of the
#   corpus predates the `months` column — so the priors read `unrecorded`, a blank is NOT 3
#   (§5 rule 2), and the Q4 is refused however well it parsed.
# ⚠️ MEASURED: CTG carried SEVEN such Q4 income statements on 2026-09-02, every one of them
#    parsed and none of them writable, blocked by a blank column in ANOTHER ROW. Re-parsing a
#    prior moves no figure — an unchanged reading goes through the merge's `fills_span`
#    branch, which writes the span and nothing else.
SPAN_OPERANDS = True

# ⚠️ ONE PROCESS PER DOCUMENT — what makes a WHOLE-TICKER run possible on a 4 GiB card.
#   True  -> `pdf_ocr_batch.run_batch`: a fresh process per filing, and it waits for the card
#            to have VRAM_FLOOR_MB free before each one.
#   False -> `pdf_ocr_job.run` parses every filing in THIS process. Right for one quarter.
# ⚠️ MEASURED 2026-09-02: 18 documents in one process cleared three filings and then every
#    `onnx@*` layer raised `CUDA failure 2: out of memory` — 294 of them — and the cascade
#    went on and reported `pdf` for statements it had been unable to read. The same 25
#    documents, one process each, ran with **0 engine errors**. It changes no semantics:
#    `seed_history` re-seeds `sane` from DISK per document and the page cache is per filing.
# ⚠️ The cost is model load, ~10-20 s per document.
ISOLATE_DOCUMENTS = True
VRAM_FLOOR_MB = 2600     # free VRAM one document wants before it starts; a filing peaked at 2.9-3.2 GiB
SHOW_ABSENT_ROWS = True  # §8 prints the rows behind a REFUSED statement — the cause, not the symptom

TEMPLATE     = None      # None = RESOLVE it (templates.csv, then CafeF's fingerprint). ⚠️ Never defaulted to "bank".
ALLOW_PARENT = True      # fall back to the STANDALONE filing where no consolidated one exists
PERIODS      = None      # the repo-native form, e.g. ["Q3-2014"]. Optional, and INTERSECTS with QUARTERS.
LAYERS       = None      # None = the cascade ONNX_ONLY selects, in cascade order
COMPARE      = True      # score every parsed cell against the statement CSV already on disk
NOTES        = ""        # free text into the run folder; blank writes a sensible default

# ── THE MERGE — section 9, one period at a time, oldest first, and UNFORCED ───────────
# ⚠️ Merging period by period is not a style choice: `merge_run` plans against disk and writes
#    afterwards, so a span recorded for one quarter reaches the NEXT quarter's planner only in
#    the following call. That is the dependency a span operand needs.
MERGE_TWO_PASS = True
MERGE_REPORTS = None     # ⚠️ WHICH STATEMENTS §9 MAY WRITE. None = all three.
                         # ⚠️ SCOPE IT WHEN YOU ARE REPAIRING ONE CELL. A quarter whose
                         # other two statements are already `pdf` FROM THE SAME filing
                         # has nothing to gain from leaving them writable, and something
                         # to lose: a DIFFERS decided by recency rather than by the
                         # filing. CTG Q4-2014 was written that way on 2026-09-03.
MERGE_APPLY   = True     # ⚠️ THE DEFAULT SINCE 2026-09-04, AND IT IS WHAT MAKES THIS
                         # NOTEBOOK WRITE. It was False, so a run that parsed perfectly
                         # ended in a PLAN and the three statement CSVs were never opened.
                         # ⚠️ MEASURED ON HOSE_FPT, 2026-09-04: a 185-minute T4 round trip
                         #    over 71 filings accepted 128 of 213 statements, §9 planned
                         #    **96 WRITEs**, and **0** of them reached disk. Two knobs had
                         #    to be flipped by hand afterwards to finish a job the machine
                         #    had already done — which is `BND-1`'s loop wearing a second
                         #    face: the work is on disk, the CSV is not, and a green run
                         #    says nothing about which.
                         #   False -> PLAN ONLY. Right when you are about to REPAIR a row,
                         #            or want to read the refusals before spending disk.
                         # ⚠️ WHAT MAKES AN AUTOMATIC WRITE DEFENSIBLE IS THE REFUSALS, NOT
                         # THE EXTRA COMMAND (CLAUDE.md §6-2-quinquadragies, which made the
                         # LOCAL per-quarter merge automatic on the same argument). §9 passes
                         # `force_differs=False`, so a figure that DIFFERS from a good `pdf`
                         # row on disk is STILL refused however `OVERWRITE` is set — and the
                         # other three refusals stand untouched: an empty `sane` band, a
                         # cumulative income statement whose priors it cannot subtract, and
                         # a document any of whose layers RAISED. A backup of the three CSVs
                         # is taken by the first call that writes anything, and every changed
                         # cell is printed. `REPAIR` in §11 is still the only way past
                         # DIFFERS, and it is still opt-in and scoped.
                         # ⚠️ AND A DRY RUN UNDERSTATES A TWO-PASS WRITE, BY CONSTRUCTION:
                         # with nothing written, a later period is planned against the span
                         # the earlier one has not recorded yet, and reports the refusal it
                         # always would. That is a property of the dry run, not a result —
                         # which is the other reason False was the worse default: it could
                         # not even tell you what True would do.

# ⚠️ REPAIR — REPLACE A `pdf` ROW THAT IS ALREADY ON DISK AND WRONG. Name the exact
#    (quarter, statement) pairs; anything not named keeps the DIFFERS refusal. The quarter is
#    the REPO-NATIVE form here:   REPAIR = [("Q3-2019", "income_statement")]
# ⚠️ `OVERWRITE = True` IS THE WRONG TOOL FOR THIS, AND THE REASON IS MEASURED. It lifts DIFFERS
#    for every statement of every quarter in the run — and a `pdf_ocr_job` run is NOT the run
#    that wrote those rows: its `sane` band is rebuilt from disk where a full `build()`
#    accumulates one as it goes, so the two escalate DIFFERENTLY and the seeded run can win on
#    an EARLIER, POORER layer. On ACB 2026-08-30 it would have replaced a 33-item balance sheet
#    with a 19-item one while repairing another statement, reporting only "DIFFERS in N columns".
# ⚠️ Read the DIFFERS report in section 7 first, and decide against the FILING (a printed
#    subtotal, the next quarter's comparative column) — never by preferring the newer run.
REPAIR = []
REPAIR_APPLY = False     # False = print the plan and change nothing. True once you agree.

EXECUTE  = True          # False = resolve and print the plan, spend nothing
REHEARSE = True          # KAGGLE only: the worker side, locally, no quota (~60 s)

## 2 · Setup — validate the parameters, find the repo

In [15]:
# ── SETUP — validate the parameters and find the repo ─────────────────────────
# ⚠️ Checked HERE, before a payload is built or a page is rendered: every one of these is a
# mistake that would otherwise surface hours later, or as a spent Kaggle round trip.
import os
import sys
from pathlib import Path

ENVIRONMENT = str(ENVIRONMENT).upper()
EXCHANGE = str(EXCHANGE).upper()
SYMBOL = str(SYMBOL).upper()
if ENVIRONMENT not in ("LOCAL", "KAGGLE"):
    raise ValueError(f"ENVIRONMENT must be 'LOCAL' or 'KAGGLE', not {ENVIRONMENT!r}")
if EXCHANGE not in ("HOSE", "HNX", "UPCOM"):
    raise ValueError(f"EXCHANGE must be HOSE, HNX or UPCOM, not {EXCHANGE!r}")

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "kaggle_gpu").is_dir()), None)
if REPO is None:
    raise RuntimeError(f"no src/kaggle_gpu at or above {Path.cwd()} — open this notebook "
                       f"from inside the repo.")
# ⚠️ `kgpu` stages the payload and talks to the Kaggle client relative to the CWD, so the
# notebook anchors itself the way a shell would. LOCAL does not need it and gets it anyway:
# one behaviour, printed, beats two that differ by a mode.
os.chdir(REPO / "src" / "kaggle_gpu")
for _p in (REPO / "src", REPO / "src" / "kaggle_gpu"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# ⚠️ A LONG-LIVED KERNEL PINS THE REPO TO THE COMMIT IT FIRST IMPORTED — `import` is a no-op
# once a module is in `sys.modules`, so re-running this notebook after the repo moves underneath
# it runs the OLD code. The loud form is an AttributeError; ⚠️ the silent form is an OCR run
# executing a previous commit's parser while `metadata.json` records HEAD's hash — a run folder
# that names code it did not run. So the repo's OWN packages are dropped here and re-imported
# from disk on every pass; third-party ones (torch, onnxruntime) are left alone, they do not
# move. ⚠️ It re-imports, so run this notebook TOP TO BOTTOM.
_OURS = ("kgpu", "utils", "web_scraper")
_RELOADED = [_n for _n in list(sys.modules) if _n.split(".")[0] in _OURS]
for _n in _RELOADED:
    del sys.modules[_n]

from utils import progress                          # noqa: E402
from web_scraper import pdf_ocr_job as job          # noqa: E402

# ⚠️ FOLDED ONCE, HERE. "2026-04" and "2026-Q4" are one quarter, and the job name, the payload
# directory and the Kaggle kernel slug are all derived from this list — two spellings that
# reached those would be two runs racing for one slug. An EMPTY list folds to `None`, which is
# `plan()`'s own contract for "every quarter this ticker files".
# ⚠️ A LIST, AND NOTHING ELSE (2026-09-03). QUARTERS used to take the strings "ALL" and
# "OUTSTANDING" beside the list, and TWO TYPES IN ONE PARAMETER COST THREE MEASURED READINGS,
# every one of which reported the wrong mistake:
#   QUARTERS = ""          an empty string is FALSY, so it fell past the sentinel test into
#                          `canonical_quarters`, which reads empty as `None` — it opened EVERY
#                          quarter this ticker files, silently, and printed "ALL".
#   QUARTERS = "  "        strips to "" and falls the same way, except `canonical_quarters`
#                          then iterates the string CHARACTER BY CHARACTER: `' ' is not a
#                          quarter`, an error about the QUARTER FORM for a mistake in the MODE.
#   QUARTERS = "2014-Q4"   the brackets forgotten — refused with `must be "ALL" or
#                          "OUTSTANDING"`, an error about the MODE for a mistake in the LIST.
# The narrowing sentinel is `ONLY_MISSING` now, the list is only ever a list, and ONE message
# covers a string, a `None` and anything else that is not one.
if not isinstance(QUARTERS, (list, tuple)):
    raise TypeError(f'QUARTERS is a LIST of quarters — [] or ["2014-Q4"] — not {QUARTERS!r}. '
                    f"{job.QUARTER_FORM}. An EMPTY list is every quarter the ticker files, "
                    f"and ONLY_MISSING = True narrows it to the ones still `missing`.")
QUARTERS = job.canonical_quarters(QUARTERS)
# ⚠️ AN EMPTY LIST IS RESOLVED IN §3, NOT HERE, and it is the only thing that is: §3 is where
# the statement CSVs and the PDF index are read, and neither has been opened yet.
RESOLVE_FROM_DISK = QUARTERS is None
OUTSTANDING_ONLY = RESOLVE_FROM_DISK and ONLY_MISSING

# ⚠️ THE CASCADE IS PART OF A RUN'S PROVENANCE, NOT ONLY OF ITS COST (`TSS-1`). `tesseract@200`
# is layer 4 of 55 HERE and does not exist on a Kaggle worker, so the two machines run
# DIFFERENT cascades and a local re-parse of a T4-parsed ticker can win on a layer the row on
# disk never saw. `ONNX_ONLY` makes the two the same 53. It never overrides an explicit
# `LAYERS`, and the resolved list is recorded in the run folder either way.
if ONNX_ONLY and LAYERS is None:
    from web_scraper.cafef_financials import FinancialsBuilder as _FB   # noqa: E402

    LAYERS = [_l.name for _l in _FB.LAYERS if _l.name.startswith("onnx")]

# ⚠️ TWO COMBINATIONS ARE REFUSED HERE RATHER THAN DISCOVERED AFTERWARDS, and both were
# measured on real runs:
#   (a) SPAN_OPERANDS needs OVERWRITE. A span operand is by definition a quarter already
#       reading `pdf`, so at OVERWRITE=False it is dropped before any OCR and the Q4 it
#       exists to unblock stays unwritable — the run would look complete and change nothing.
#   (b) OVERWRITE + MERGE_INTO_CSV passes `force_differs=True` into the automatic per-quarter
#       merge, i.e. it lifts DIFFERS for EVERY statement of every quarter in the run. On ACB
#       (2026-08-30) that would have replaced a 33-item balance sheet with a 19-item one while
#       repairing a different statement. §9's merge is unforced; REPAIR is the scoped escape.
if SPAN_OPERANDS and not OVERWRITE:
    raise ValueError("SPAN_OPERANDS needs OVERWRITE = True — a span operand is a quarter "
                     "already reading `pdf`, and OVERWRITE=False drops it before any OCR.")
if OVERWRITE and MERGE_INTO_CSV:
    raise ValueError("OVERWRITE = True passes force_differs into the automatic merge, which "
                     "lifts DIFFERS for every statement of the run. Leave MERGE_INTO_CSV off "
                     "and use §9 (unforced, one period at a time), or REPAIR for one row.")

# The task label every progress line carries. ONE string, built once: LOCAL replaces it per
# document (`doc 2/3 HOSE_TCB Q3-2013`), KAGGLE keeps it for all six steps.
# ⚠️ Rebuilt in §3 when the sentinel resolves: a label reading "all quarters" over a
# three-quarter run is a progress line lying about its own denominator.
LABEL = f"{EXCHANGE}_{SYMBOL} " + (" ".join(QUARTERS) if QUARTERS else "all quarters")

print(f"environment : {ENVIRONMENT}")
print(f"ticker      : {EXCHANGE}_{SYMBOL}")
print("quarters    : " + ("OUTSTANDING — resolved in §3 from what is on disk"
                          if OUTSTANDING_ONLY else
                          f"{QUARTERS or 'ALL — every quarter this ticker files'}"))
# ⚠️ A KNOB THAT IS NOT READ HAS TO SAY SO WHERE IT WOULD BE READ. Silence is what lets a reader
# believe a flag they set had an effect, and this one is inert the moment the list names its own
# quarters.
if QUARTERS and ONLY_MISSING:
    print("            : ⚠️ ONLY_MISSING is IGNORED — it is read only when QUARTERS is "
          "empty, and this run names its quarters.")
print(f"overwrite   : {OVERWRITE}"
      + ("" if OVERWRITE else "   (quarters already `pdf` in all three are skipped)"))
print(f"upsert csv  : {MERGE_INTO_CSV}"
      + ("   per quarter, as each finishes" if MERGE_INTO_CSV and ENVIRONMENT == "LOCAL"
         else "   after the pull" if MERGE_INTO_CSV else ""))
print(f"bootstrap   : {FORCE_EMPTY_BAND}"
      + ("   an EMPTY `sane` band is written anyway — the only way a new ticker "
         "starts" if FORCE_EMPTY_BAND else "   an EMPTY `sane` band is REFUSED"))
print(f"isolation   : "
      + ("one process per document, VRAM floor "
         f"{VRAM_FLOOR_MB} MiB   (`GPU-1`)" if ISOLATE_DOCUMENTS and ENVIRONMENT == "LOCAL"
         else "one process for the whole run" if ENVIRONMENT == "LOCAL" else "n/a — KAGGLE"))
print(f"cascade     : "
      + (f"{len(LAYERS)} layer(s)" if LAYERS else "the full cascade")
      + ("   onnx only — the cascade a Kaggle worker runs (`TSS-1`)"
         if ONNX_ONLY and LAYERS else ""))
print(f"repo        : {REPO}")
print(f"cwd         : {Path.cwd()}")
print(f"code        : {REPO / 'src'}"
      + (f"   ({len(_RELOADED)} cached module(s) dropped, re-imported from disk)"
         if _RELOADED else "   (first import in this kernel)"))
# ⚠️ The percentage is a POSITION IN THE PLAN and not a fraction of the time left — a filing
# accepted at layer 1 of 47 costs ~1 min and one that defeats the cascade cost 33. Said here,
# once, because it is on every line below it.
print(f"log shape   : {progress.format_line(0.337, 'task', 'sub-task', 'detail')}"
      f"   ← overall %, a position in the plan")

environment : KAGGLE
ticker      : HOSE_FPT
quarters    : ALL — every quarter this ticker files
overwrite   : True
upsert csv  : False
bootstrap   : True   an EMPTY `sane` band is written anyway — the only way a new ticker starts
isolation   : n/a — KAGGLE
cascade     : 67 layer(s)   onnx only — the cascade a Kaggle worker runs (`TSS-1`)
repo        : d:\GIT\master-thesis
cwd         : d:\GIT\master-thesis\src\kaggle_gpu
code        : d:\GIT\master-thesis\src   (9 cached module(s) dropped, re-imported from disk)
log shape   :  33.7% - task - sub-task - detail   ← overall %, a position in the plan


## 3 · What is left — the gap on disk, and what a re-run cannot change

In [16]:
# ── WHAT IS LEFT — the gap on disk, and what a re-run cannot change ───────────
# ⚠️ THE QUESTION THIS ANSWERS IS THE ONE THAT DECIDES `QUARTERS`, and until 2026-09-02 the
# notebook could not answer it: you had to know which (quarter, statement) cells of this ticker
# still read `missing`, and the only way to find out was an ad-hoc script over the three CSVs.
#
# ⚠️ IT IS NOT A SECOND RULE. The quarters come from `documents()` through `job.plan()` — the
# same call the run makes — "already done" is `job.parsed_reports()`, which is `pdf` and nothing
# else, and a cell a past run PROVED unproducible is dropped by `settled_absences`.
#
# ⚠️ `use_data_root()` FIRST, AND IT IS LOAD-BEARING (`CWD-1`). `fin.STATEMENTS_DIR` is a
# RELATIVE default read at call time, and §2 has just `os.chdir`-ed into `src/kaggle_gpu` — so
# without this every quarter reads `absent`, which is a legitimate state for a ticker being
# bootstrapped and therefore looks like nothing is wrong.
from web_scraper import cafef_financials as fin      # noqa: E402
from web_scraper import pdf_ocr_batch                # noqa: E402

job.use_data_root(REPO / "raw_data" / "cafef")
_builder = fin.FinancialsBuilder(logger=None)

# ⚠️ RESOLVED, NEVER DEFAULTED — and how it resolved is printed, because "read off
# templates.csv" and "fingerprinted over the network" are not the same claim (`TPX-1`).
[PLAN] = pdf_ocr_batch.plan_batch(
    [SYMBOL], exchange=EXCHANGE, reports_root=REPO / "reports" / "pdf_ocr",
    allow_parent=ALLOW_PARENT, span_operands=SPAN_OPERANDS, template=TEMPLATE,
    builder=_builder)
TEMPLATE, TEMPLATE_HOW = PLAN.template, PLAN.template_how

print(f"{PLAN.key}   template {TEMPLATE} ({TEMPLATE_HOW})   "
      f"{PLAN.filed} quarter(s) filed, {PLAN.complete} complete")
print("")
# ⚠️ NO FILINGS AND NOTHING OUTSTANDING PRINT THE SAME LINE OTHERWISE, and they are opposite
# answers: one says the ticker is done, the other that nothing was ever measured (§5 rule 2).
if not PLAN.filed:
    print("  ⚠️ this ticker files NO document `documents()` will open — an absent PDF index, or")
    print("     everything before FINANCIALS_PERIOD_MIN. Nothing here says the ticker is done.")
elif not PLAN.quarters and not PLAN.settled:
    print("  every filed quarter reads `pdf` in all three statements. Nothing is outstanding.")
else:
    for _q in PLAN.quarters:
        _tag = "SPAN OPERAND — re-parsed only to record `months`" if _q in PLAN.operands else \
               "open — a re-run could still win it"
        print(f"  {_q:9} {_tag}")
    for _q, _reports in sorted(PLAN.settled.items()):
        for _r in _reports:
            print(f"  {_q:9} {_r:18} SETTLED — the filing contains no such statement")
    print("")
    print(f"  {len(PLAN.quarters)} quarter(s) with an OPEN cell "
          f"(of which {len(PLAN.operands)} are span operands), "
          f"{sum(len(v) for v in PLAN.settled.values())} SETTLED cell(s)")

# ⚠️ A SETTLED CELL IS `missing` FOREVER, and re-running it costs the full cascade to return the
# same word. ACB's Q2-2009 and Q3-2009 cash flows were put through all 50 layers FOUR times on
# 2026-08-30 before anything recorded why: both filings are three-page `BÁO CÁO TÀI CHÍNH TÓM
# TẮT` forms (Mẫu CBTT-03) with no cash flow statement in them at all.
# ⚠️ AND AN EMPTY SETTLED SET IS SILENCE, NOT A CLEAN BILL: a run older than artefact schema v4
# recorded no reason, so a cell reading "open" here may still be unwinnable and merely
# unmeasured (§5 rule 2).
if PLAN.settled:
    print("")
    print("  `missing` is the correct and PERMANENT answer for the SETTLED rows (§5 rule 24).")

# ⚠️ WHICH QUARTERS THE RUN ACTUALLY TAKES — three modes, and an EMPTY `QUARTERS` is resolved
# HERE and nowhere else. An ONLY_MISSING that resolves to nothing RAISES rather than falling
# through: `plan()` reads an empty `quarters` as "every quarter this ticker files", so the one
# thing a "nothing left to do" answer must not do is silently open 70 filings.
if OUTSTANDING_ONLY:
    if not PLAN.quarters:
        raise RuntimeError(
            f"ONLY_MISSING resolved to nothing for {PLAN.key}: every filed quarter either "
            f"reads `pdf` in all three statements or is SETTLED. Name the quarters explicitly, "
            f"or set ONLY_MISSING = False, if you meant to re-parse something anyway.")
    QUARTERS = PLAN.quarters
elif RESOLVE_FROM_DISK:
    # ⚠️ EVERY QUARTER THE TICKER FILES — including the ones already `pdf`, which is the point:
    # this is the mode that gets a ticker to FULL coverage rather than filling its gaps. It
    # needs OVERWRITE (validated in §2) and, on this card, ISOLATE_DOCUMENTS.
    QUARTERS = [job.as_quarter(t.period) for t in
                job.plan(_builder, EXCHANGE, SYMBOL, allow_parent=ALLOW_PARENT,
                         template=TEMPLATE)]
    PLAN.quarters = QUARTERS
else:
    # ⚠️ `else`, not `elif QUARTERS`: an empty list is the two branches above, so everything
    # reaching here NAMES its quarters. A truthiness test would leave a fourth, silent path that
    # ran with `PLAN.quarters` still holding §3's outstanding set — a run taking quarters nobody
    # asked for, with nothing printing a difference.
    PLAN.quarters = list(QUARTERS)

LABEL = f"{EXCHANGE}_{SYMBOL} " + (" ".join(PLAN.quarters[:4]) + (" …" if len(PLAN.quarters) > 4
                                                                  else "")
                                   if PLAN.quarters else "all quarters")
print("")
_MODE = "OUTSTANDING" if OUTSTANDING_ONLY else "ALL" if RESOLVE_FROM_DISK else "explicit"
print(f'  QUARTERS = {_MODE} -> {len(PLAN.quarters)} document(s)'
      + (f": {' '.join(PLAN.quarters)}" if len(PLAN.quarters) <= 12 else
         f": {' '.join(PLAN.quarters[:6])} … {' '.join(PLAN.quarters[-3:])}"))

HOSE_FPT   template corp (detect_template (CafeF fingerprint, over the network))   71 quarter(s) filed, 0 complete

  2008-Q3   open — a re-run could still win it
  2008-Q4   open — a re-run could still win it
  2009-Q1   open — a re-run could still win it
  2009-Q2   open — a re-run could still win it
  2009-Q3   open — a re-run could still win it
  2009-Q4   open — a re-run could still win it
  2010-Q1   open — a re-run could still win it
  2010-Q2   open — a re-run could still win it
  2010-Q3   open — a re-run could still win it
  2010-Q4   open — a re-run could still win it
  2011-Q1   open — a re-run could still win it
  2011-Q2   open — a re-run could still win it
  2011-Q3   open — a re-run could still win it
  2011-Q4   open — a re-run could still win it
  2012-Q1   open — a re-run could still win it
  2012-Q2   open — a re-run could still win it
  2012-Q3   open — a re-run could still win it
  2012-Q4   open — a re-run could still win it
  2013-Q1   open — a re-run could stil

## 4 · The plan — what would run, before anything is spent

In [17]:
# ── THE JOB — resolved and printed, before anything is spent ──────────────────
# ⚠️ Both branches end at the SAME object. `pdf_ocr.job()` writes a `JobSpec`'s fields into the
# worker notebook's parameter cell, and the worker builds the JobSpec from them — so a LOCAL run
# and a KAGGLE run of the same parameters are one procedure on two machines, not two. What
# differs is the stack, and every run records its `stack_fingerprint`.
SPEC = CFG = PREPARED = None

if ENVIRONMENT == "LOCAL":
    SPEC = job.JobSpec(
        exchange=EXCHANGE, symbol=SYMBOL, periods=PERIODS, quarters=PLAN.quarters,
        allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
        compare_with_disk=COMPARE, merge_into_csv=MERGE_INTO_CSV,
        force_empty_band=FORCE_EMPTY_BAND,
        notes=NOTES or f"ENVIRONMENT=LOCAL overwrite={OVERWRITE}",
    )
    # ⚠️ `prepare()` resolves the data root, the models, the TEMPLATE and the document list and
    # RAISES on any of them — no OCR, no PDF. It also raises, in as many words, when every
    # quarter you asked for is already parsed and OVERWRITE is False.
    PREPARED = SPEC.prepare()
    print("\n".join(PREPARED.describe()))
    print()
    for _t in PREPARED.tasks:
        print(f"  {_t.period:<8} {_t.file[:56]:<56} "
              f"{os.path.getsize(_t.path) / 1024 ** 2:>6.1f} MB"
              + ("  CUMULATIVE" if _t.cumulative else ""))
    # ⚠️ THE CEILING, BEFORE ANY OF IT IS SPENT. The bill is `pages x OCR passes`, and the 49
    # layers are only 7 passes — a layer that changes only the mapping or a gate re-maps a parse
    # the page cache already holds. Both numbers are free: `page_count` opens the PDF without
    # rendering a pixel, and the pass count is a property of the cascade.
    # ⚠️ It is a CEILING, loose in the honest direction: `scan` stops as soon as all three
    # statements are behind it (BID Q3-2011 reads 7 pages of 32) and the cascade stops at the
    # first layer that accepts. What it tells you is which filing would be dear IF something in
    # it cannot be read — that is the only case that pays it.
    import fitz                                       # noqa: E402
    from web_scraper.cafef_financials import ocr_key  # noqa: E402

    PASSES = len({ocr_key(_l) for _l in PREPARED.layers})
    PAGES = 0
    for _t in PREPARED.tasks:
        try:
            with fitz.open(_t.path) as _d:
                PAGES += _d.page_count
        except Exception as _e:                       # a damaged page tree is `scan`'s problem
            print(f"  ⚠️ could not count pages of {_t.file}: {_e}")
    print("")
    print(f"  ceiling      : {PAGES} page(s) x {PASSES} OCR pass(es) = "
          f"{PAGES * PASSES:,} page-reads at most")
    print(f"                 ~{PAGES * PASSES * 0.65 / 60:.0f} min at 0.65 s/page "
          f"(onnx@200 on this laptop; the 300/400 dpi passes cost more).")
    print("                 A filing accepted at layer 1 pays ONE pass over the pages "
          "up to its last")
    print("                 statement, which is the usual case — see the run log.")

    if PREPARED.template != "bank":
        print(f"\n⚠️ CRP-1: this is a `{PREPARED.template}` filing. `C_LIABILITIES` still "
              f"misses on corp,\n   so the balance sheet reconciles on the TRIVIAL "
              f"`assets == resources` — true by\n   construction on any page that reads both. "
              f"Nothing from a non-bank run may be\n   quoted as a fundamental yet.")
else:
    from kgpu import pdf_ocr, runner                 # noqa: E402

    CFG = pdf_ocr.job(
        SYMBOL, exchange=EXCHANGE, periods=PERIODS, quarters=PLAN.quarters,
        allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
        compare=COMPARE, notes=NOTES, merge_statements=MERGE_INTO_CSV,
        # ⚠️ NOT a worker parameter. The worker cannot upsert — it writes /kaggle/working and
        # exits — so this is the PULL's knob, read by `runner.merge_statements` on this machine.
        force_empty_band=FORCE_EMPTY_BAND,
    )
    print("\n".join(pdf_ocr.describe(CFG)))
    print()
    # The filings this selects are the filings the WORKER will open: `plan()` runs HERE, so the
    # payload cannot diverge from the worker's own choice.
    runner.plan(CFG)

job          : pdf-ocr-fpt-2008-q3-2026-q1
kernel       : ductrung180200/mt-pdf-ocr-fpt-2008-q3-2026-q1
dataset      : ductrung180200/mt-cafef-filings-fpt-2008-q3-2026-q1
filings      : HOSE_FPT  periods=all  quarters=['2008-Q3', '2008-Q4', '2009-Q1', '2009-Q2', '2009-Q3', '2009-Q4', '2010-Q1', '2010-Q2', '2010-Q3', '2010-Q4', '2011-Q1', '2011-Q2', '2011-Q3', '2011-Q4', '2012-Q1', '2012-Q2', '2012-Q3', '2012-Q4', '2013-Q1', '2013-Q2', '2013-Q3', '2013-Q4', '2014-Q1', '2014-Q2', '2014-Q3', '2014-Q4', '2015-Q1', '2015-Q2', '2015-Q3', '2015-Q4', '2016-Q1', '2016-Q2', '2016-Q3', '2016-Q4', '2017-Q1', '2017-Q2', '2017-Q3', '2017-Q4', '2018-Q1', '2018-Q2', '2018-Q3', '2018-Q4', '2019-Q1', '2019-Q2', '2019-Q3', '2019-Q4', '2020-Q1', '2020-Q2', '2020-Q3', '2020-Q4', '2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4', '2022-Q1', '2022-Q2', '2022-Q3', '2022-Q4', '2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4', '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4', '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4', '2026-Q1'] 

## 5 · Rehearse — KAGGLE only: the worker side, locally, no quota

In [18]:
# ── STAGE + REHEARSE — KAGGLE only: the worker side, locally, no quota ────────
# ⚠️ THE PAYLOAD IS STAGED HERE, AND IT HAS TO BE: a rehearsal runs the worker against
# `.payload/<job>/`, so there is nothing to rehearse until that exists. `export` is local and
# free — it writes the zip, it does not upload; the RUN cell below re-exports and uploads, so
# nothing here commits you to anything.
# ⚠️ The rehearsal runs no OCR pass. What it proves is that the payload holds every input the
# parse reads, under BOTH of Kaggle's mount layouts, and it prints the magnitude band `sane`
# will get. AN EMPTY BAND IS THE WARNING TO STOP FOR: `sane` fails open without one, and that
# is the documented way a run writes a wrong figure (CLAUDE.md §6-2-octodecies).
if ENVIRONMENT == "KAGGLE" and REHEARSE:
    from kgpu import export                       # noqa: E402

    # Two steps, one line each, in the same shape the RUN cell prints — `capture()` re-emits
    # `export`'s and `rehearse`'s own output as the DETAIL of the step that produced it.
    DRESS = progress.Stages([("export", "stage payload", 1.0),
                             ("rehearse", "rehearse worker", 1.0)], label=LABEL)
    DRESS.begin("export", "local, no upload, no quota")
    with DRESS.capture():
        export.export(CFG)                        # -> .payload/<job>/  (no upload)
    DRESS.begin("rehearse", "both Kaggle mount layouts")
    with DRESS.capture():
        runner.rehearse(CFG)
    DRESS.done("rehearsed — nothing was spent")
else:
    print("skipped" if ENVIRONMENT == "KAGGLE" else "LOCAL — nothing to rehearse")


  0.0% - step 1/2 HOSE_FPT 2008-Q3 2008-Q4 2009-Q1 2009-Q2 … - stage payload - local, no upload, no quota
  0.0% - step 1/2 HOSE_FPT 2008-Q3 2008-Q4 2009-Q1 2009-Q2 … - stage payload - 71 filing(s) of HOSE_FPT (Q3-2008, Q4-2008, Q1-2009, Q2-2009, Q3-2009, Q4-2009, Q1-2010, Q2-2010, Q3-2010, Q4-2010, Q1-2011, Q2-2011, Q3-2011, Q4-2011, Q1-2012, Q2-2012, Q3-2012, Q4-2012, Q1-2013, Q2-2013, Q3-2013, Q4-2013, Q1-2014, Q2-2014, Q3-2014, Q4-2014, Q1-2015, Q2-2015, Q3-2015, Q4-2015, Q1-2016, Q2-2016, Q3-2016, Q4-2016, Q1-2017, Q2-2017, Q3-2017, Q4-2017, Q1-2018, Q2-2018, Q3-2018, Q4-2018, Q1-2019, Q2-2019, Q3-2019, Q4-2019, Q1-2020, Q2-2020, Q3-2020, Q4-2020, Q1-2021, Q2-2021, Q3-2021, Q4-2021, Q1-2022, Q2-2022, Q3-2022, Q4-2022, Q1-2023, Q2-2023, Q3-2023, Q4-2023, Q1-2024, Q2-2024, Q3-2024, Q4-2024, Q1-2025, Q2-2025, Q3-2025, Q4-2025, Q1-2026)
  0.0% - step 1/2 HOSE_FPT 2008-Q3 2008-Q4 2009-Q1 2009-Q2 … - stage payload - documents.zip: 92 files, 451.8 MB (511.4 MB uncompressed)
  0.0% - step

## 6 · Run

In [19]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# ⚠️ One line shape on both machines — ` 33.7% - <task> - <sub-task> - <detail>`, one formatter
# (`utils.progress`), so the two cannot drift. LOCAL the task is the DOCUMENT and the sub-task
# its position in the cascade; KAGGLE the task is the STEP of the round trip.
# ⚠️ THE OVERALL % IS A POSITION IN THE PLAN, NOT A FRACTION OF THE TIME. A filing accepted at
# its FIRST OCR pass is ~1 min and one that defeats all 24 of them was 33, so the number is a
# LOWER BOUND — a run finishes early, it does not stall at 99 %.
#
# ⚠️ **`ISOLATE_DOCUMENTS` IS WHAT MAKES A WHOLE-TICKER RUN POSSIBLE ON THIS CARD.** Measured
# 2026-09-02: an 18-document run inside ONE process cleared three filings and then every
# `onnx@*` layer raised `CUDA failure 2: out of memory` — 294 of them — and the cascade went on
# and reported `pdf` for statements it had been unable to read. `pdf_ocr_batch.run_batch` spawns
# one process per document and waits for the card to have `VRAM_FLOOR_MB` free before each; the
# same 25 documents then ran with **0 engine errors**. It changes no semantics, because
# `seed_history` re-seeds `sane` from DISK per document and `PdfParser._ocr_cache` is scoped to
# one filing — see the module docstring.
FOLDERS: list = []
LATEST = EXIT = REPORT = None

if not EXECUTE:
    print("EXECUTE = False — the plan above is resolved and nothing was spent")
elif ENVIRONMENT == "LOCAL" and ISOLATE_DOCUMENTS:
    from web_scraper import pdf_ocr_batch                # noqa: E402

    FOLDERS = pdf_ocr_batch.run_batch(
        [PLAN], layers=LAYERS, allow_parent=ALLOW_PARENT, overwrite=OVERWRITE,
        compare=COMPARE, notes=NOTES or f"{EXCHANGE}_{SYMBOL} — one process per document",
        vram_floor_mb=VRAM_FLOOR_MB)
elif ENVIRONMENT == "LOCAL":
    # ⚠️ THE OLD PATH, AND IT IS KEPT FOR ONE DOCUMENT AT A TIME. `job.run` parses every planned
    # filing in THIS process, which is right for a repair of one quarter and is what died at
    # document 4 of 18. It prints the progress line itself and writes the SAME line into the run
    # folder's `run.log`, so what you read here is what a later reader gets.
    LATEST = job.run(SPEC)
    FOLDERS = [LATEST]
else:
    if MERGE_INTO_CSV:
        print("MERGE_INTO_CSV is on: accepted statements are upserted into\n"
              "    raw_data/.../statements/ after the pull, with a backup taken first\n"
              "    and every changed cell printed.")
    # `refresh_data=True` re-exports and re-uploads the payload every time — correct, because
    # the filter above may have changed since the last run of this job.
    REPORT = progress.Stages(runner.RUN_STAGES, label=LABEL)
    EXIT = runner.run(CFG, refresh_data=True, progress=REPORT)
    REPORT.note(f"exit {EXIT}   (0 = COMPLETE and pulled)")

  0.0% - step 1/6 HOSE_FPT 2008-Q3 2008-Q4 2009-Q1 2009-Q2 … - stage payload - pdf-ocr-fpt-2008-q3-2026-q1
  0.0% - step 1/6 HOSE_FPT 2008-Q3 2008-Q4 2009-Q1 2009-Q2 … - stage payload - 71 filing(s) of HOSE_FPT (Q3-2008, Q4-2008, Q1-2009, Q2-2009, Q3-2009, Q4-2009, Q1-2010, Q2-2010, Q3-2010, Q4-2010, Q1-2011, Q2-2011, Q3-2011, Q4-2011, Q1-2012, Q2-2012, Q3-2012, Q4-2012, Q1-2013, Q2-2013, Q3-2013, Q4-2013, Q1-2014, Q2-2014, Q3-2014, Q4-2014, Q1-2015, Q2-2015, Q3-2015, Q4-2015, Q1-2016, Q2-2016, Q3-2016, Q4-2016, Q1-2017, Q2-2017, Q3-2017, Q4-2017, Q1-2018, Q2-2018, Q3-2018, Q4-2018, Q1-2019, Q2-2019, Q3-2019, Q4-2019, Q1-2020, Q2-2020, Q3-2020, Q4-2020, Q1-2021, Q2-2021, Q3-2021, Q4-2021, Q1-2022, Q2-2022, Q3-2022, Q4-2022, Q1-2023, Q2-2023, Q3-2023, Q4-2023, Q1-2024, Q2-2024, Q3-2024, Q4-2024, Q1-2025, Q2-2025, Q3-2025, Q4-2025, Q1-2026)
  0.0% - step 1/6 HOSE_FPT 2008-Q3 2008-Q4 2009-Q1 2009-Q2 … - stage payload - documents.zip: 92 files, 451.8 MB (511.4 MB uncompressed)
  0.0% - ste

## 7 · The result — verdicts from the run folder

In [20]:
# ── READ THE RUN FOLDERS ────────────────────────────────────────────────
# ⚠️ Read back from disk rather than from anything in memory, so this measures what a later
# reader would actually get. `metadata.json` already carries the whole scorecard in `results`.
# ⚠️ **A BATCH IS MANY FOLDERS, ONE PER DOCUMENT.** `FOLDERS` comes from the run cell; when the
# kernel was restarted between the two, fall back to this ticker's folders newer than the
# newest CSV backup — never to "the newest folder" alone, which on a batch is the LAST document
# and would report a 70-quarter run as a one-quarter one.
import json                                          # noqa: E402

PATTERN = f"*__{EXCHANGE.lower()}_{SYMBOL.lower()}__pdf_ocr"
if not FOLDERS:
    FOLDERS = sorted((REPO / "reports" / "pdf_ocr").glob(PATTERN), key=lambda p: p.name)[-1:]
LATEST = FOLDERS[-1] if FOLDERS else None
META = MERGE = None
RESULTS: list = []

if LATEST is None:
    print(f"no run folder matching {PATTERN}")
else:
    META = json.loads((LATEST / "metadata.json").read_text(encoding="utf-8"))
    inputs, ocr = META.get("inputs", {}), META.get("environment", {}).get("ocr", {})
    SCHEMA = META.get("schema_version", 1)
    print(f"{len(FOLDERS)} run folder(s), {FOLDERS[0].name} … {LATEST.name}")
    print(f"  commit       : {META.get('git_commit')}")
    print(f"  template     : {inputs.get('template')}  ({inputs.get('template_how')})")
    # ⚠️ THE TWO OCR HALVES FAIL INDEPENDENTLY — detection is onnxruntime, recognition is torch
    # — so "the GPU was used" is two questions. `ORT-1` is a green run that was half on the CPU
    # because onnxruntime ADVERTISED a provider the session then could not create.
    print(f"  detection    : {(ocr.get('det_providers') or ['?'])[0]}"
          f"   (onnxruntime {ocr.get('onnxruntime')})")
    print(f"  recognition  : {ocr.get('recognizer_device')}")
    print(f"  stack        : {ocr.get('stack_fingerprint')}"
          + (f"   ⚠️ PIN VIOLATIONS: {ocr['pin_violations']}"
             if ocr.get("pin_violations") else ""))

    # ⚠️ **A RUN WHOSE ONNX LAYERS RAISED REPORTS `pdf` WITH A REAL LAYER AND A REAL ITEM
    # COUNT.** The rows below look identical to a good run; what happened is that the layer
    # could not run, the cascade went on, and something later won BY DEFAULT — which on this
    # machine is `tesseract@200`, layer 4 of 55. Measured 2026-09-02 on HOSE_CTG: 85 layers
    # raised `CUDA failure 2: out of memory` and 30 of 33 statements were reported `pdf`.
    # ⚠️ `pdf_ocr_merge` refuses such a document whole (`VCR-1`), so nothing reaches disk — but
    # that is the LAST line of defence and it is silent about WHY until §8. This says it here,
    # where the verdict table is read.
    RAISED = {}
    for _folder in FOLDERS:
        for _doc in sorted((_folder / "documents").glob("*.json")):
            _d = json.loads(_doc.read_text(encoding="utf-8"))
            if _d.get("engine_errors"):
                RAISED[_d.get("period", _doc.stem)] = _d["engine_errors"]
        _m = json.loads((_folder / "metadata.json").read_text(encoding="utf-8"))
        RESULTS += _m.get("results", [])
    if RAISED:
        print()
        print(f"  ⚠️ {len(RAISED)} document(s) had at least one layer RAISE rather than refuse.")
        print("     Whatever won them won BY DEFAULT, and the merge refuses them whole.")
        for _p in sorted(RAISED)[:8]:
            print(f"       {_p:10} {len(RAISED[_p])} layer(s): "
                  f"{', '.join(l for l, _ in RAISED[_p][:3])}")
        _kinds = sorted({str(w).split(";")[0].strip()[:70]
                         for e in RAISED.values() for _l, w in e})
        for _k in _kinds[:3]:
            print(f"       cause: {_k}")
        print("     ⚠️ `out of memory` means the card was short — raise VRAM_FLOOR_MB, close "
              "other")
        print("        CUDA processes, and re-run those quarters. Nothing of theirs is on disk.")

    print()
    print(f"  {'period':10} {'report':18} {'layer':30} {'items':>5}  {'status':8} verdict")
    for r in sorted(RESULTS, key=lambda r: (fin._period_key(r["period"]), r["report"])):
        print(f"  {r['period']:10} {r['report']:18} {(r['layer'] or '—'):30} "
              f"{r['items']:>5}  {r['status']:8} {r['verdict']}")
    # ⚠️ `seconds` is the DOCUMENT's cost repeated on each of its three report rows, so it is
    # summed per PERIOD. A set would also collapse two documents that took the same time.
    PER_DOC = {r["period"]: r["seconds"] for r in RESULTS}
    _ok = sum(1 for r in RESULTS if r["status"] == "pdf")
    print(f"\n  parse: {sum(PER_DOC.values()) / 60:.1f} min over {len(PER_DOC)} document(s)"
          f"   {_ok} of {len(RESULTS)} statement(s) accepted")

1 run folder(s), 20260904-020205__hose_fpt__pdf_ocr … 20260904-020205__hose_fpt__pdf_ocr
  commit       : 65d839b9+dirty
  template     : corp  (override)
  detection    : CUDAExecutionProvider   (onnxruntime 1.22.0)
  recognition  : cuda
  stack        : 88df8ef02c08

  period     report             layer                          items  status   verdict
  Q3-2008    balance_sheet      —                                  0  absent   absent in this run
  Q3-2008    cash_flow          onnx@200                          23  pdf      no pdf row on disk to compare against
  Q3-2008    income_statement   —                                  0  absent   absent in this run
  Q4-2008    balance_sheet      —                                  0  absent   absent in this run
  Q4-2008    cash_flow          onnx@200+relax                    17  pdf      no pdf row on disk to compare against
  Q4-2008    income_statement   —                                  0  absent   absent in this run
  Q1-2009    bala

## 8 · Refused vs written — two questions, two places

In [21]:
# ── WHAT WAS REFUSED, AND WHAT WAS WRITTEN ───────────────────────────────
#   the PARSE refused a statement   -> the document JSON's `absent_reasons`, and `run.log`
#   the MERGE refused a statement   -> the `merge` block, written by whatever ran the UPSERT
# ⚠️ ON KAGGLE THOSE ARE TWO MACHINES. A cell that greps the worker's `run.log` for
# `WRITE `/`skip ` finds nothing on a Kaggle run and, finding nothing, used to print "no
# refusals — every statement was accepted". That false success was printed over a run that
# wrote 0 of 201 accepted cells (HOSE_CTG, 2026-08-30).
#
# ⚠️ **THE REASON IS DATA NOW, NOT PROSE** (`absent_reasons`, artefact schema v4) — and since
# 2026-09-02 so are the ROWS behind it (`absent_rows`). A reason names the SYMPTOM (`no total
# assets`); the rows say WHAT THE FILING PRINTS where the chart expects that anchor, which is
# the only thing a fix can be written from. Recovering that used to cost a second OCR run.
if FOLDERS:
    print("── the PARSE refused ────────────────────────────────────────")
    ABSENT: dict = {}
    for _folder in FOLDERS:
        for _doc in sorted((_folder / "documents").glob("*.json")):
            _d = json.loads(_doc.read_text(encoding="utf-8"))
            for _rep, _tried in (_d.get("absent_reasons") or {}).items():
                ABSENT[(_d["period"], _rep)] = (_tried, (_d.get("absent_rows") or {}).get(_rep))
    if not ABSENT:
        print("  nothing — every statement the cascade opened was accepted")
    for (_period, _rep), (_tried, _rows) in sorted(
            ABSENT.items(), key=lambda kv: (fin._period_key(kv[0][0]), kv[0][1])):
        print(f"  {_period:9} {_rep:18}")
        for _layer, _why in _tried:
            print(f"      [{_layer:28}] {_why}")
        # ⚠️ THE ROWS ARE THE CAUSE AND THE REASON IS THE SYMPTOM. Printed only for the
        # statements this run could not accept, and only the EARLIEST reading of them — the
        # last layer is always the most relaxed one and its rows answer a question nobody
        # asked (§6-2-duovicies' trap for the reason applies to the rows too).
        if _rows and SHOW_ABSENT_ROWS:
            print(f"      rows read at [{_rows['layer']}], pages {_rows['pages']}, "
                  f"{len(_rows['rows'])} row(s) — the ones naming a TOTAL:")
            for _r in _rows["rows"]:
                _lab = (_r["label"] or "").upper()
                if any(w in _lab for w in ("TỔNG", "TONG", "CUỐI", "CUOI", "ĐẦU", "DAU")):
                    print(f"        {_r['key'][:54]:54} {_r['values'][:2]}")
                    print(f"          {_r['label'][:96]}")

    # ⚠️ NOT a refusal — a fact about the FILING. A page whose scan is turned reads as vertical
    # noise, and before 2026-08-30 that cost a whole statement in silence (BID Q3-2011).
    TURNED = [ln for _f in FOLDERS
              for ln in (_f / "run.log").read_text(encoding="utf-8",
                                                   errors="replace").splitlines()
              if "text lines are vertical" in ln]
    if TURNED:
        print("\n── pages the READ had to turn ──────────────────────────────")
        for ln in TURNED[:12]:
            print("  " + progress.detail_of(ln))

    print("\n── the MERGE decided ───────────────────────────────────────")
    EVENTS = []
    for _folder in FOLDERS:
        _m = json.loads((_folder / "metadata.json").read_text(encoding="utf-8"))
        for _ev in (_m.get("merge") or {}).get("events", []):
            EVENTS += [(d, _ev["applied"]) for d in _ev["decisions"]]
    if EVENTS:
        for _d, _applied in sorted(EVENTS, key=lambda e: (fin._period_key(e[0]["period"]),
                                                          e[0]["report"])):
            mark = "WRITE " if _d["action"] == "write" else "skip  "
            items = f"[{_d['layer']}] {_d['items']} items" if _d["layer"] else ""
            print(f"  {mark} {_d['period']:9} {_d['report']:18} {items:32} {_d['reason']}")
            # ⚠️ A CAVEAT ON A WRITE IS LOUDER THAN A REFUSAL, because a refusal stops and a
            # write does not. Printed only for a WRITE: refusal 1 sets the note before
            # refusals 2-4 have had their say, so beside `skip` it would contradict the line.
            if _d.get("note") and _d["action"] == "write":
                print(f"           ⚠️  {_d['note']}")
        _w = sum(1 for d, a in EVENTS if d["action"] == "write" and a)
        print(f"\n  -> {_w} statement(s) written, {len(EVENTS) - _w} refused or planned only")
        if not _w:
            print("  ⚠️ NOTHING REACHED raw_data/. On a ticker with no CSV yet the commonest "
                  "reason is an")
            print("     EMPTY `sane` band — set FORCE_EMPTY_BAND = True. It lifts ONE guard "
                  "and no other,")
            print("     so screen the artefact before quoting anything (`BND-1`).")
    else:
        print("  ⚠️ NO MERGE RAN against these run folders — the statement CSVs were not "
              "opened.")
        print("     §9 below does it: MERGE_TWO_PASS = True, then MERGE_APPLY = True.")

── the PARSE refused ────────────────────────────────────────
  Q3-2008   balance_sheet     
      [onnx@200                    ] reconcile: assets != liabilities + equity
      rows read at [onnx@200], pages [3, 4, 5, 12, 13, 15], 75 row(s) — the ones naming a TOTAL:
        tong_cong_tai_san                                      [270, None]
          TỔNG CỘNG TÀI SẢN
        von_dau_tu_cua_chu_so_huu                              [411, None]
          Vốn đầu tư của chủ sở hữu
        quy_du_tru_bo_sung_von_dl_dau_tu_phat_trien            [417, None]
          Quỹ dự trữ bổ sung vốn ĐL & đầu tư phát triển
        nguon_von_dau_tu_xay_dung_co_ban                       [421, None]
          Nguồn vốn đầu tư xây dựng cơ bản
        tong_cong_nguon_von                                    [600, None]
          TỔNG CỘNG NGUỒN VỐN
        dau_nay_tu_ke_nam_luy                                  [42458648356, 40573602318]
          đầu nay từ kế Năm Luỹ
        tong_cong_tien_cac_cac_khoan_tuon

## 9 · The merge — one period at a time, oldest first, and UNFORCED

In [ ]:
# ── THE MERGE — one period at a time, oldest first, and UNFORCED ──────────
# ⚠️ WHY NOT ONE CALL OVER THE FOLDER: `merge_run` runs `plan_merge` against disk FIRST and
# `_write` afterwards, so every decision in one call is taken against the SAME disk state.
# `_quarter_priors` reads a prior's `months` from that state, so the span a Q3 records reaches
# Q4's planner only in the NEXT call. That is `SPN-1`'s dependency, and it is why a batch that
# re-parses a span operand AND the Q4 it unblocks must merge them separately, oldest first.
# ⚠️ AND NOTHING HERE LIFTS A REFUSAL. `force_differs` is not passed, so a reading that
# disagrees with disk is refused exactly as it would be by default — which is the honest
# outcome, not a failure of this cell. `REPAIR` in §11 is the scoped escape.
# ⚠️ ONE BACKUP PER TICKER, taken by the first call that actually writes.
from web_scraper import pdf_ocr_batch                  # noqa: E402

if not MERGE_TWO_PASS:
    print("MERGE_TWO_PASS = False — nothing was merged.")
elif not FOLDERS:
    print("no run folder to merge — run the cells above first.")
else:
    print(f"       force_differs=False   force_empty_band={FORCE_EMPTY_BAND}")
    print(f"       reports={MERGE_REPORTS or 'all three'}")
    print()
    TALLY = pdf_ocr_batch.merge_batch(
        FOLDERS, apply=MERGE_APPLY, reports=MERGE_REPORTS,
        force_empty_band=FORCE_EMPTY_BAND)
    print()
    if not MERGE_APPLY:
        print("nothing was written. Set MERGE_APPLY = True to apply the plan above.")
        print("⚠️ AND THE PLAN ABOVE UNDERSTATES IT, BY CONSTRUCTION: with nothing written, a")
        print("   later period is planned against a span the earlier one has not recorded yet,")
        print("   and reports the refusal it always would. A dry run cannot show a second pass")
        print("   that depends on the first.")
    elif TALLY["written"]:
        print(f"{TALLY['written']} statement(s) reached raw_data/.../statements/ — §10 reads")
        print("the CSVs themselves, which is the only place the two can be told apart.")
    else:
        # ⚠️ "THE RUN FINISHED" AND "THE CSV CHANGED" ARE DIFFERENT FACTS, and only the
        # second was ever the point. Read back from each folder's own `merge` block — the
        # structured record `record_merge` has just written — rather than from the lines
        # above, so this reports what a later reader gets and not what this cell printed.
        import collections                                # noqa: E402

        print("⚠️ NOTHING REACHED raw_data/.../statements/. Every accepted statement was")
        print("   refused, and these are the refusals, most common first:")
        WHY = collections.Counter(
            (_d["reason"] or "").split(" — ")[0].split(" because ")[0][:64]
            for _f in FOLDERS
            for _ev in (json.loads((Path(_f) / "metadata.json").read_text(encoding="utf-8"))
                        .get("merge") or {}).get("events", []) if _ev["applied"]
            for _d in _ev["decisions"] if _d["action"] != "write")
        for _reason, _n in (WHY.most_common(6)
                            or [("(no merge block — nothing was planned)", 0)]):
            print(f"     {_n:>4}  {_reason}")
        # ⚠️ THE ONE REFUSAL THAT CLOSES ON ITSELF, and the only one a knob here lifts.
        if any("band" in _r for _r in WHY):
            print("   ⚠️ `sane` band EMPTY is `BND-1`, and it is a LOOP: this ticker has no")
            print("      `pdf` row on disk, so `seed_history` builds no magnitude band, so")
            print("      every statement is refused, so there is still no CSV.")
            print("      FORCE_EMPTY_BAND = True is the only way out of it — and it LIFTS A")
            print("      REAL GUARD, so screen the figures by arithmetic first (two statements")
            print("      agreeing on one figure, a printed subtotal closing) before quoting")
            print("      any of them.")

## 10 · Did it land? — the statement CSVs themselves

In [23]:
# ── DID IT LAND? — the statement CSVs themselves ───────────────────────────
# ⚠️ Everything above reports what some process DECIDED; this reads what is on disk. The two
# came apart on a Kaggle round trip that finished green, wrote a complete run folder and created
# no CSV at all (`BND-1`, HOSE_BSR and again HOSE_CTG) — and no amount of log reading would have
# said so, because the merge that refused everything ran on the other machine.
import csv                                            # noqa: E402

from web_scraper import cafef_financials as fin       # noqa: E402

# ⚠️ `CWD-1`, AND THIS CELL WALKED STRAIGHT INTO IT. `statement_path()` reads
# `fin.STATEMENTS_DIR` at call time and its module default is RELATIVE — while the SETUP cell
# `os.chdir`s to `src/kaggle_gpu`, where `kgpu` stages its payload. So the first version of this
# cell reported `NO FILE` for a ticker whose three CSVs were on disk. The path is PRINTED,
# because a directory nobody names is a directory nobody checks.
ROOT = job.use_data_root(job.DEFAULT_DATA_ROOT)

TPL = (META or {}).get("inputs", {}).get("template") or TEMPLATE
# ⚠️ KEYED BY (period, REPORT), not by period. A merge writes one statement of a quarter and
# skips another — VCB Q1-2026 wrote its income statement and cash flow while its balance sheet
# was `identical to the row already on disk` — so a period-only set credits this run with a row
# it deliberately left alone.
MINE = {(d["period"], d["report"])
        for ev in (MERGE or {}).get("events", [])
        for d in ev["decisions"] if d["action"] == "write" and ev["applied"]}
if TPL is None:
    print("no template resolved — run the cells above first")
else:
    print(f"{ROOT / 'financials' / 'statements' / TPL}   {EXCHANGE}_{SYMBOL}")
    print()
    ANY = False
    for _report in fin.REPORTS:
        _path = Path(fin.statement_path(TPL, _report, EXCHANGE, SYMBOL))
        if not _path.is_file():
            print(f"  {_report:18} ⚠️ NO FILE — {_path.name} does not exist")
            continue
        ANY = True
        with open(_path, encoding="utf-8-sig") as _f:
            _rows = list(csv.DictReader(_f))
        _src = {}
        for _r in _rows:
            _src[_r.get("source", "")] = _src.get(_r.get("source", ""), 0) + 1
        _mine = [_r for _r in _rows if _r.get("source") == "pdf"
                 and (_r["period"], _report) in MINE]
        print(f"  {_report:18} {len(_rows):>3} quarters   "
              + "  ".join(f"{k}={v}" for k, v in sorted(_src.items()))
              + (f"   <- {len(_mine)} from this run" if _mine else ""))
        # ⚠️ Rule 24: a financial statement comes from the filing PDF and from nothing else.
        # A `cafef` row is an HTML transcription and must not be in this file.
        if _src.get("cafef"):
            print(f"       ⚠️ {_src['cafef']} row(s) read `source=cafef` — an HTML "
                  f"transcription. §5 rule 24 forbids it.")
    if not ANY:
        print()
        print("  ⚠️ THIS TICKER HAS NO STATEMENT CSV AT ALL. The parse is in the run folder "
              "and")
        print("     nothing was upserted — the MERGE section above says which refusal "
              "stopped it.")


D:\GIT\master-thesis\raw_data\cafef\financials\statements\corp   HOSE_FPT

  balance_sheet        1 quarters   pdf=1
  income_statement     1 quarters   missing=1
  cash_flow            1 quarters   pdf=1


## 11 · Repair one row — scoped, deliberate, read the diff first

In [24]:
# ── REPAIR ONE ROW — scoped, deliberate, and read the diff first ──────────
# ⚠️ THE ONLY WAY THIS NOTEBOOK OVERWRITES A GOOD-LOOKING `pdf` ROW. `pdf_ocr_merge` refuses a
# figure that DIFFERS from a `pdf` row on disk, because two runs disagreeing is not settled by
# preferring the newer one. That refusal is lifted here for the NAMED pairs only — never for the
# run — and `merge_run`'s own `periods`/`reports` filter is what scopes it.
# ⚠️ A BACKUP is taken before any write. Diff EVERY COLUMN afterwards, not the figures: three
# separate runs in this repo lost only a `publish_date` and a figures-only diff called each of
# them clean (CLAUDE.md §6-2-quatervicies, §6-2-quinvicies, §6-2-quadragies).
if LATEST is not None and REPAIR:
    from web_scraper import pdf_ocr_merge                 # noqa: E402

    HOW = "APPLY" if REPAIR_APPLY else "PLAN"
    print(f"{HOW} — {len(REPAIR)} scoped repair(s) from {LATEST.name}")
    print()
    for _period, _report in REPAIR:
        print(f"── {_period} {_report} " + "─" * 46)
        _rep = pdf_ocr_merge.merge_run(
            LATEST, apply=REPAIR_APPLY, periods=[_period], reports=[_report],
            force_differs=True, force_empty_band=FORCE_EMPTY_BAND)
        if getattr(_rep, "backup", None):
            print(f"   backup: {_rep.backup}")
    if not REPAIR_APPLY:
        print()
        print("nothing was written. Set REPAIR_APPLY = True to apply the plan above.")
elif LATEST is not None:
    print("REPAIR is empty — no row already on disk was replaced.")
    print("  A statement this run parsed that disk already holds as `pdf` was refused as")
    print("  DIFFERS and left alone. That is the default and usually right; name the")
    print("  (quarter, statement) pair in REPAIR only once the FILING has settled which")
    print("  reading is correct.")


REPAIR is empty — no row already on disk was replaced.
  A statement this run parsed that disk already holds as `pdf` was refused as
  DIFFERS and left alone. That is the default and usually right; name the
  (quarter, statement) pair in REPAIR only once the FILING has settled which
  reading is correct.
